# Exploration de données temporelles
**Données**: température et précipitations horaires de Pointe-au-Père (PaP) et de Mont-Joli (MJ), débit quotidien de la Petite rivière Neigette, mesures aux 15 minutes de la station météo du Mont-Comi.

In [ ]:
# Bloc important les librairies nécessaires au laboratoire
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pyhomogeneity import pettitt_test
from statsmodels.graphics.tsaplots import plot_acf
from outils import lire, moyennes, matrice, ccf, preblanchir, verifier, mann_kendall, date_fr
pap, mj, ne, co = lire("PaP"), lire("MJ"), lire("Neigette"), lire("Comi")

## Bloc 1. Visualisation des données
La première étape quand on veut utiliser des données est de les visualiser. Ce bloc sert à produire des figures qui permettent de le faire pour mieux connaitre les données.

### 1a. Station météorologique du Mont-Comi
Mesures chaque 15 minutes, du 25 septembre au 6 novembre 2026.

In [ ]:
temperatures = ["temp_air_C", "temp_surface_sol_C", "temp_sol_C"]
autres = ["hr_pct", "pluie_mm", "neige_cm", "vent_dir_deg", "vent_vit_ms", "ray_incident_Wm2"]
fig, ax = plt.subplots(len(autres) + 1, 1, figsize=(11, 13), sharex=True)
for v in temperatures:
    ax[0].plot(co.index, co[v], lw=.7, label=v)
ax[0].set_ylabel("température (°C)"); ax[0].legend(ncol=3, fontsize=9)
for a, v in zip(ax[1:], autres):
    if v == "pluie_mm": a.bar(co.index, co[v], width=15 / 1440)
    else: a.plot(co.index, co[v], lw=.6)
    a.set_ylabel(v)
plt.tight_layout();

In [ ]:
def matrice_jour_heure(s):
    """Matrice d'une série aux 15 minutes."""
    return s.to_frame("v").assign(jour=s.index.date, heure=s.index.hour + s.index.minute / 60).pivot_table(index="jour", columns="heure", values="v")

fig, ax = plt.subplots(1, 3, figsize=(17, 6), sharey=True)
for a, v, cmap in [(ax[0], "temp_air_C", "magma"), (ax[1], "ray_incident_Wm2", "cividis"), (ax[2], "pluie_mm", "Blues")]:
    Mv = matrice_jour_heure(co[v])
    im = a.imshow(Mv, aspect="auto", cmap=cmap, interpolation="nearest", extent=[0, 24, len(Mv), 0])
    fig.colorbar(im, ax=a, label=v); a.set_xlabel("heure (UTC)"); a.set_xticks(range(0, 25, 3))
ax[0].set_yticks(range(0, len(Mv), 7), [str(j) for j in Mv.index[::7]]);

> **ATTENTION**: définissez les paramètres de la cellule suivante, puis exécutez-la.

In [ ]:
# PARAMÈTRES
VARIABLE = None   # variable de la matrice: "temp_surface_sol_C", "temp_sol_C", "hr_pct" ou "vent_vit_ms"
verifier(VARIABLE=VARIABLE)

M = matrice_jour_heure(co[VARIABLE])
plt.figure(figsize=(9, 6))
plt.imshow(M, aspect="auto", cmap="magma", interpolation="nearest", extent=[0, 24, len(M), 0])
plt.colorbar(label=VARIABLE); plt.xlabel("heure (UTC)"); plt.xticks(range(0, 25, 3))
plt.yticks(range(0, len(M), 7), [str(j) for j in M.index[::7]]);

In [ ]:
moy = co[["ray_incident_Wm2", "temp_air_C", "temp_surface_sol_C", "temp_sol_C"]].groupby(co.index.hour + co.index.minute / 60).mean()
moy48 = pd.concat([moy, moy.set_axis(moy.index + 24)])                    # même cycle moyen répété sur 48 h
h0 = moy.ray_incident_Wm2.idxmax()                                         # maximum du rayonnement (jour 1)
fen = moy48[(moy48.index >= h0) & (moy48.index < h0 + 24)]                 # maxima cherchés dans les 24 h qui suivent
fig, ax = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
for a, v in zip(ax, moy48):
    a.plot(moy48.index, moy48[v]); a.axvline(fen[v].idxmax(), color="r", ls="--"); a.set_ylabel(v)
T = ["temp_air_C", "temp_surface_sol_C", "temp_sol_C"]
for a in ax[1:]: a.set_ylim(moy[T].min().min() - .3, moy[T].max().max() + .3)   # même échelle pour les trois températures
ax[-1].set_xlabel("heure (UTC), sur deux jours"); ax[-1].set_xticks(range(0, 49, 6))
ax[0].set_title("Cycle journalier moyen répété sur 48 h (trait rouge = maximum)")
plt.tight_layout()
pd.DataFrame({"heure du maximum (UTC)": (fen.idxmax() % 24), "retard sur le rayonnement (h)": fen.idxmax() - h0,
              "amplitude moyenne": (moy.max() - moy.min()).round(2)})

### 1b. Débit de la petite rivière Neigette
Voici un exemple de code pour générer une figure permettant de visualiser rapidement les données de débit journalier de la petite rivière Neigette.

In [ ]:
from matplotlib.colors import LogNorm
M = matrice(ne.debit_m3s)
plt.figure(figsize=(9, 4.5))
plt.imshow(M, aspect="auto", cmap="viridis", interpolation="nearest", norm=LogNorm(),   # couleurs en échelle logarithmique
           extent=[1, 366, M.index.max() + .5, M.index.min() - .5])
plt.colorbar(label="débit (m³/s)", format="%g"); plt.xlabel("jour de l'année"); plt.ylabel("année");

## Bloc 2. Détection de tendances
### 2a. Préparation des séries temporelles

> **ATTENTION**: définissez les paramètres de la cellule suivante, puis exécutez-la.

In [ ]:
# PARAMÈTRES
COUVERTURE = None   # proportion des données disponibles pour qu'un mois soit pris en compte (p.ex. 0.5 = 50% des données, 1 = 100% des données)
verifier(COUVERTURE=COUVERTURE)

Tm = moyennes(pap.temp_air_C, "MS", COUVERTURE)                       # moyennes mensuelles
Ta = Tm.resample("YS").mean()[Tm.resample("YS").count() == 12]        # année valide = 12 mois valides
print(f"{Tm.notna().sum()} mois valides, {len(Ta)} années valides ({Ta.index.year.min()}–{Ta.index.year.max()})")

### 2b. Vérification des prémisses du test de Mann-Kendall

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
Tm.plot(ax=ax[0], lw=.7, title="Moyennes mensuelles"); ax[0].set_ylabel("°C")
plot_acf(Ta.values, lags=8, ax=ax[1], title="Autocorrélation des moyennes annuelles");

### 2c. Choix et application du test

> **ATTENTION**: définissez les paramètres de la cellule suivante, puis exécutez-la.

In [ ]:
# PARAMÈTRES
AGREGATION = None   # "annuelle" ou "mensuelle"
TEST = None         # "original", "hamed_rao" (retire l'autocorrélation), "saisonnier" (retire la saisonnalité), "saisonnier_correle" (retire les deux)
verifier(AGREGATION=AGREGATION, TEST=TEST)

r = mann_kendall(Ta if AGREGATION == "annuelle" else Tm, TEST)
par_an = AGREGATION == "annuelle" or TEST.startswith("saisonnier")   # pente exprimée par an, sinon par mois
print(f"Tendance : {r.trend} ; p = {r.p:.3f} ; pente de Sen = {r.slope * (10 if par_an else 120):.2f} °C/décennie")

### 2d. Analyse de sensibilité des paramètres

In [ ]:
lignes = []
for c in [0.5, 0.8, 0.95]:
    tm = moyennes(pap.temp_air_C, "MS", c); ta = tm.resample("YS").mean()[tm.resample("YS").count() == 12]
    for agr, test, s, fac in [("annuelle", "original", ta, 10), ("mensuelle", "saisonnier_correle", tm, 10)]:
        r_ = mann_kendall(s, test)
        lignes.append([c, agr, test, s.notna().sum(), round(r_.p, 3), round(r_.slope * fac, 2)])
pd.DataFrame(lignes, columns=["couverture", "agrégation", "test", "n", "p", "Sen (°C/déc.)"])

## Bloc 3. Détection du décalage temporel dans les précipitations entre deux stations météorologiques (Pointe-au-Père et Mont-Joli)
Corrélation croisée des précipitations horaires 2016–2025 (les stations sont à une distance d'environ 25 km).

> **ATTENTION** : définissez les paramètres de la cellule suivante, puis exécutez-la.

In [ ]:
# PARAMÈTRES
MOIS = None          # "mai-oct" ou "annee"
PREBLANCHIR = None   # True ou False : retirer l'autocorrélation avant de corréler
verifier(MOIS=MOIS, PREBLANCHIR=PREBLANCHIR)

def correlation(mois, preblanchir_):
    P = pd.concat({"PaP": pap.precip_mm, "MJ": mj.precip_mm}, axis=1)["2016":"2025"].dropna()
    if mois == "mai-oct": P = P[P.index.month.isin(range(5, 11))]
    x, y = (preblanchir(P.PaP, P.MJ)[:2] if preblanchir_ else (P.PaP, P.MJ))
    return ccf(x, y)

c = correlation(MOIS, PREBLANCHIR)
plt.bar(c.index, c.values); plt.xlabel("décalage (h) ; > 0 = MJ en retard"); plt.ylabel("corrélation");

In [ ]:
pd.DataFrame({f"{m}, préblanchie={pb}": correlation(m, pb)[[-1, 0, 1]].round(2)
              for m in ["mai-oct", "annee"] for pb in [False, True]}).T.rename(columns=lambda k: f"décalage {k:+d} h")

## Bloc 4. Extraction du début de la crue printanière
Le test de Pettitt permet de tester l'existence d'**une seule** rupture et sa position.

> **ATTENTION** : définissez les paramètres de la cellule suivante, puis exécutez-la.

In [ ]:
# PARAMÈTRES
FENETRE_DEBUT = None   # début de la fenêtre, format "mm-jj" (mois-jour) : p. ex. "02-01" = 1er février
FENETRE_FIN = None     # fin de la fenêtre, format "mm-jj" (p. ex. "06-30" = 30 juin) ou "pic" (jour du débit maximal entre mars et juin)
verifier(FENETRE_DEBUT=FENETRE_DEBUT, FENETRE_FIN=FENETRE_FIN)

def fenetre(an, deb, fin):
    """Débit de l'année `an` entre deb et fin ("pic" = jour du débit maximal entre mars et juin)."""
    s = ne.debit_m3s[f"{an}-{deb}":f"{an}-{'06-30' if fin == 'pic' else fin}"].interpolate(limit=3)
    if s.isna().mean() > 0.1 or len(s) < 30: return None          # fenêtre trop incomplète ou trop courte
    s = s.dropna()
    if fin == "pic": s = s[:s[f"{an}-03-01":].idxmax()]
    return s if len(s) >= 10 else None

def debuts_crue(deb, fin):
    dates = {}
    for an in sorted(set(ne.index.year)):
        s = fenetre(an, deb, fin)
        if s is not None: dates[an] = s.index[pettitt_test(s.values, sim=None).cp].dayofyear
    return pd.Series(dates)

s = fenetre(2019, FENETRE_DEBUT, FENETRE_FIN)
r = pettitt_test(s.values, sim=None)
s.plot(title=f"Exemple 2019 : rupture le {date_fr(s.index[r.cp])}")
plt.axvline(s.index[r.cp], color="r", ls="--"); plt.ylabel("débit (m³/s)"); plt.show()
print(f"p-value du test de Pettitt : {r.p:.1e}")
print(f"Autocorrélation du débit entre deux jours consécutifs : {s.autocorr(1):.2f}")

In [ ]:
d = debuts_crue(FENETRE_DEBUT, FENETRE_FIN)
r = mann_kendall(d, "original")
d.plot(marker="o", title="Début de la crue printanière")
reperes = [32, 46, 60, 74, 91, 105, 121, 135]                             # 1er et 15 de chaque mois, de février à mai
plt.yticks(reperes, [date_fr(j) for j in reperes]); plt.ylim(d.min() - 7, d.max() + 7); plt.xlabel("année"); plt.show()
print(f"Mann-Kendall : tendance = {r.trend}, p = {r.p:.2f}")
print(f"Pente de Sen : {r.slope*10:.1f} jours par décennie (valeur négative = crue de plus en plus précoce)")

In [ ]:
lignes = []
for deb in ["02-01", "03-01"]:
    for fin in ["06-30", "pic"]:
        d_ = debuts_crue(deb, fin); r_ = mann_kendall(d_, "original")
        lignes.append([date_fr(deb), date_fr(fin), date_fr(d_.mean()), round(d_.std(), 1), date_fr(d_.min()), round(r_.p, 2)])
pd.DataFrame(lignes, columns=["début fenêtre", "fin fenêtre", "date moyenne", "écart-type (jours)", "date la plus précoce", "p Mann-Kendall"])